# 0. Configuration

In [17]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize,Bounds,minimize_scalar
import statsmodels

import matplotlib.pyplot as plt
import seaborn as sns

# model preparation
from sklearn.model_selection import train_test_split

# modeling
import xgboost as xgb

# evaluation
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss


# 1. Load Data

In [2]:
search_df = pd.read_csv('search_marketplace.csv')
search_df.head()

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,4/4/2013 8:32,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,4/4/2013 8:32,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,4/4/2013 8:32,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,4/4/2013 8:32,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,4/4/2013 8:32,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


# 2. Train Test Split

In [12]:
X= search_df.drop(columns=['click_bool','gross_bookings_usd','booking_bool','date_time'])
y = search_df['click_bool']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [13]:
X_train

,srch_id,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,prop_brand_bool,...,comp5_rate_percent_diff,comp6_rate,comp6_inv,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff
153248,10271,5,219,NaN,NaN,219,109257,2,4.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
67802,4525,5,219,NaN,NaN,219,47167,3,5.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
148889,9983,16,31,NaN,NaN,31,15920,2,4.5,0,...,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103093,6898,5,35,3.46,72.16,35,84978,4,4.5,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
104681,7011,5,219,NaN,NaN,100,132210,4,4.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,7993,5,219,NaN,NaN,219,54075,3,3.5,1,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
103694,6944,32,220,NaN,NaN,219,21315,3,4.5,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN
131932,8825,5,219,NaN,NaN,219,135390,2,4.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146867,9853,14,100,NaN,NaN,219,2906,3,4.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN


In [14]:
y_train.info()

<class 'pandas.core.series.Series'>
Index: 160000 entries, 153248 to 121958
Series name: click_bool
Non-Null Count   Dtype
--------------   -----
160000 non-null  int64
dtypes: int64(1)
memory usage: 2.4 MB


In [15]:
# Train in the clear and quantize the weights
model = xgb.XGBClassifier()
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [16]:
eval_test=model.predict(X_test)

In [18]:
# roc; useful for false positives
auc = roc_auc_score(y_test, eval_test)
ap  = average_precision_score(y_test, eval_test)
ll  = log_loss(y_test, eval_test)

print(f"Test ROC-AUC: {auc:.4f}")
print(f"Test Avg Precision (PR-AUC): {ap:.4f}")

Test ROC-AUC: 0.5071
Test Avg Precision (PR-AUC): 0.0494
